In [2]:
using LinearAlgebra
using BoundaryValueDiffEq
using Plots
using DifferentialEquations
using BSplineKit
using DelimitedFiles

In [ ]:
##similar method##

In [ ]:
function sol_baseflowODE(tspan,Num)

    function oneDiskODE!(du, u , p, t)
        
        U = u[1]
        dU = u[2]
        V = u[3]
        dV = u[4]
        W = u[5]
        du[1] = dU
        ddU = U^2 + W*dU - (V+1.0e0)^2
        du[2] = ddU
        ddV = 2.0e0*U*(V + 1.0e0) + W*dV
        du[3] = dV
        du[4] = ddV                          
        du[5] = -2.0e0*U

    end
    function oneDiskbc!(residual, u , p, t)

        residual[1] = u[begin][1] 
        residual[2] = u[begin][3] 
        residual[3] = u[begin][5] 
        residual[4] = u[end][1] 

        residual[5] = u[end][3] + 1.0e0
    end
        prob = BVProblem(oneDiskODE!, oneDiskbc!, [0.0, 0.5103341351120374, 0.0, -0.6151547026271073, 0.0] ,tspan, dtmax=0.01)
        sol = solve(prob, Shooting(Vern7()), dt=0.001)
        t=range(0.0, 20, Num)
        u=sol(t)
    
    return u , t

end


In [ ]:
function velocity(u)

    U = u[1 , :]
    dU = u[2 , :]
    V = -1 * u[3 , :]
    dV = -1 * u[4 , :]
    W = u[5 , :]
    phi = 0.001 .* cumsum(U)
    N = itp = interpolate(t, U , BSplineOrder(4))
    N1 = itp = interpolate(t, dU , BSplineOrder(4))
    N2 = itp = interpolate(t, dV , BSplineOrder(4))
    N3 = itp = interpolate(t, phi , BSplineOrder(4))
    
    return N,N1,N2,N3,U,V,W,phi

end

In [ ]:
function sol_q(tspan,Num,initial_vec)
   
    function com_von_q(du,u,p,t)
        q = u[1]
        dq = u[2]
        du[1] = u[2]
        du[2] = -2 * sigma * N3(t) * u[2]
        
    end
    function bc1!(residual,u,p,t)
        residual[1] = u[begin][1] - 1
        residual[2] = u[end][1] 
        
    end
        prob = BVProblem(com_von_q, bc1!, initial_vec,tspan)
        sol = solve(prob, MIRK4(), dt=0.001)
        t = range(0.0, 20, Num)
        u = sol(t)
        q = u[1, :]
        dq = u[2,:]

    return q , dq

end

In [ ]:
function sol_f(tspan,Num,initial_vec)
    function com_von_f(du,u,p,t)
        f = u[1]
        df = u[2]
        du[1] = u[2]
        du[2] = -2 * sigma * N3(t) * u[2] + 2 * sigma * N(t) *u[1] + 2 * sigma * (N1(t)^2 + N2(t)^2) 
    end
    function bc2!(residual,u,p,t)
        residual[1] = u[begin][1] - 0
        residual[2] = u[end][1] - 0
    end
        prob = BVProblem(com_von_f, bc2!, initial_vec,tspan)
        sol = solve(prob, MIRK4(), dt=0.001)
        t = range(0.0, 20, Num)
        u = sol(t)
        f = u[1, :]
        df = u[2,:]

        return f , df
        
end

In [ ]:
function com_von_T(f ,q ,Mx ,gamma ,Tw)
    T = 1 .-(gamma-1)/2 * Mx^2 .* f .+ (Tw - 1) .*q
    return T
end
##
function com_von_W(phi,T)
    W = -2 * T .* phi
end

In [ ]:
function caculate_z(t :: StepRangeLen , gamma , Mx , Tw , f , q)
    start = t.ref.hi
    step = t.step.hi
    len = t.len
    z = zeros(len , 1)
    integ_f = 0
    integ_q = 0
    for i = 1 : len
        z[i,1] = t[i,1] - (gamma - 1) / 2 * Mx^2 * integ_f + (Tw - 1) * integ_q
        integ_f = f[i,1] * step + integ_f
        integ_q = q[i,1] * step + integ_q
    end
    return z
end

In [39]:
tspan = (0,20)
Num=20001
initial_vec = [0,0]
sigma = 0.7
Mx = 8
gamma = 1.4
u,t = sol_baseflowODE(tspan,Num)
N,N1,N2,N3,U,V,W,phi = velocity(u)
q , dq = sol_q(tspan,Num,initial_vec)
f , df = sol_f(tspan,Num,initial_vec)
data = t
for Tw = 0 : 0.4 : 2
    T = com_von_T(f ,q ,Mx ,gamma ,Tw)
    z = caculate_z(t , gamma , Mx , Tw , f , q)
    data = [data z T]
end

In [ ]:
##CFD method##


In [1]:
function Cheb(N)

    θ = range(0,length=N+1,stop=pi)
    x = reshape(-cos.(θ), N+1, 1)
    c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
    X = repeat(x, 1, N+1);
    dX = X - X';
    D = (c * (1 ./ c)') ./ (dX .+ I(N+1));
    D = D - diagm(vec(sum(D, dims=2))); 
    return D,x
    
end 

Cheb (generic function with 1 method)